<a href="https://colab.research.google.com/github/andilMc/gemmafro-e2b/blob/main/notebooks/07_demo_gradio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 07 — Démo live Gradio (Gemma E2B + LoRA)

Interface de test en direct

In [ ]:
# Installe les dépendances (gradio pour l'interface, le reste comme 04/05).
!pip install -q -U transformers accelerate peft bitsandbytes gradio

In [ ]:
# Monte Drive et localise l'adaptateur LoRA final produit en Phase 3.
import os
from google.colab import drive

drive.mount('/content/drive')  # demande l'autorisation d'accès au Drive

PROJECT_DIR = '/content/drive/MyDrive/gemmafro-e2b'  # racine du projet sur Drive
ADAPTER_DIR = f'{PROJECT_DIR}/checkpoints/gemma-4-e2b-lora-final'  # adaptateur LoRA final de la Phase 3
assert os.path.isdir(ADAPTER_DIR), "Adaptateur introuvable — exécuter 03_finetune.ipynb jusqu'au bout d'abord."

In [ ]:
# Se connecte à Hugging Face avec le token des Colab Secrets.
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get('HF_TOKEN'))  # authentifie la session avec le token HF_TOKEN

In [ ]:
# Charge le modèle de base en 4-bit et lui rattache l'adaptateur LoRA — identique à 04_evaluate.ipynb et
# 05_generate_submission.ipynb, pour garantir exactement le même comportement que ce qui a été évalué.
#
# Nous chargeons ensuite une DEUXIÈME instance complète du modèle de base (sans adaptateur), pour la
# comparaison zero-shot. Ce n'est pas redondant : `model.disable_adapter()` est une bascule globale
# sur un objet PeftModel partagé, donc pas thread-safe — deux threads qui généreraient en même temps
# sur le même `model` (l'un avec l'adaptateur actif, l'autre l'ayant désactivé) se marcheraient dessus
# et corrompraient les deux sorties. Deux modèles séparés en mémoire éliminent complètement ce risque
# et permettent une vraie génération simultanée. Coût : environ le double de VRAM pour le modèle de
# base (~6-7 Go x2) — nécessite un GPU L4 (24 Go) ou plus, un T4 (16 Go) serait trop juste.
import gc
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

MODEL_NAME = "google/gemma-4-E2B-it"
MAX_SEQ_LENGTH = 512  # même longueur maximale qu'à l'entraînement

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)  # tokenizer Gemma
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token  # pas de token de padding défini : on réutilise le token de fin

gc.collect()  # libère la mémoire (Python, puis cache GPU)
torch.cuda.empty_cache()

bnb_config = BitsAndBytesConfig(  # même quantification 4-bit qu'à l'entraînement
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

def load_base_model():
    """Charge une instance fraîche du modèle de base quantifié — appelée deux fois (fine-tuné et
    zero-shot) pour obtenir deux objets modèle totalement indépendants."""
    m = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        torch_dtype=torch.bfloat16,
        device_map={"": 0},  # tout sur le GPU 0, sans offload CPU
    )
    m.config.pad_token_id = tokenizer.pad_token_id
    m.config.use_cache = True  # inférence : le cache KV accélère la génération
    return m

def unwrap_clippable_linears(model):
    """Même déballage qu'en Phase 3/4/5 : nécessaire pour que PeftModel retrouve la structure
    de modules sur laquelle l'adaptateur a été entraîné. Appliqué aussi au modèle zero-shot pour
    rester cohérent avec la façon dont le zero-shot est mesuré partout ailleurs dans le projet
    (04_evaluate.ipynb compare aussi sur le modèle déballé, pas sur les couches natives)."""
    count = 0
    for module in model.modules():
        for child_name, child in list(module.named_children()):
            if child.__class__.__name__ == "Gemma4ClippableLinear":
                setattr(module, child_name, child.linear)
                count += 1
    print(f'{count} couches Gemma4ClippableLinear déballées')
    return model

# Modèle 1 : base + adaptateur LoRA (réponse fine-tunée)
base_model = unwrap_clippable_linears(load_base_model())
model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)  # rattache les poids LoRA au modèle de base
model.eval()  # mode inférence (désactive le dropout)

# Modèle 2 : base seul, jamais touché par un adaptateur (réponse zero-shot)
base_model_zeroshot = unwrap_clippable_linears(load_base_model())
base_model_zeroshot.eval()

gc.collect()
torch.cuda.empty_cache()
print(f"Mémoire GPU allouée (les deux modèles) : {torch.cuda.memory_allocated() / 1e9:.2f} Go")

In [ ]:
# Même template de prompt qu'en Phases 2/3/4/5 — mais sans sélecteur de langue dans l'interface :
# en Phase 2 (02_build_dataset.ipynb), la langue passée à build_prompt() était toujours celle de la
# question elle-même (subset_to_language_name(row['subset'])), jamais une langue cible différente.
# Le modèle n'a donc jamais appris à traduire vers une langue distincte de celle de la question — un
# sélecteur qui le laisserait croire était trompeur (l'instruction était simplement ignorée quand
# elle ne correspondait pas à la langue réelle de la question). On aligne donc le prompt sur ce que
# le modèle sait réellement faire : répondre dans la même langue que la question posée.
def build_prompt(question: str) -> str:
    return (
        "Réponds à la question de santé suivante dans la même langue que la question, "
        f"de façon claire et médicalement fiable.\n\nQuestion : {question}"
    )

In [ ]:
# Génération en streaming (mot par mot), et lancement simultané des deux modèles sur deux threads
# indépendants — plus d'attente de l'un avant de commencer l'autre.
import time
import queue
from threading import Thread
from transformers import TextIteratorStreamer

def _start_stream(target_model, chat_prompt, max_new_tokens=400):
    """Démarre la génération d'un modèle dans un thread séparé et renvoie son streamer (déjà en
    train de se remplir en arrière-plan) — ne bloque pas, contrairement à un simple appel .generate()."""
    inputs = tokenizer(chat_prompt, return_tensors='pt', add_special_tokens=False).to(target_model.device)
    streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)
    gen_kwargs = dict(
        **inputs, max_new_tokens=max_new_tokens, do_sample=False,
        pad_token_id=tokenizer.pad_token_id, streamer=streamer,
    )

    def _run():
        with torch.no_grad():
            target_model.generate(**gen_kwargs)

    thread = Thread(target=_run)
    thread.start()
    return streamer, thread

def dual_stream(chat_prompt, compare, max_new_tokens=400):
    """Lance la génération fine-tunée (et, si demandé, zero-shot) sur deux threads distincts, sur
    deux modèles séparés — voir cellule précédente pour pourquoi un seul modèle partagé ne peut pas
    faire ça en toute sécurité. Cède (ft_text, zs_text, ft_done, zs_done) à chaque nouveau morceau
    de texte reçu de N'IMPORTE LEQUEL des deux threads, sans attendre l'autre.
    """
    streamer_ft, thread_ft = _start_stream(model, chat_prompt, max_new_tokens)
    streams = {'ft': streamer_ft}
    threads = {'ft': thread_ft}
    if compare:
        streamer_zs, thread_zs = _start_stream(base_model_zeroshot, chat_prompt, max_new_tokens)
        streams['zs'] = streamer_zs
        threads['zs'] = thread_zs

    partial = {'ft': '', 'zs': ''}
    done = {'ft': False, 'zs': not compare}

    while not (done['ft'] and done['zs']):
        progressed = False
        for key, streamer in streams.items():
            if done[key]:
                continue
            try:
                chunk = streamer.text_queue.get(timeout=0.02)  # court timeout : on alterne entre les deux sans bloquer
            except queue.Empty:
                continue
            if chunk == streamer.stop_signal:  # sentinelle interne du streamer : cette génération est terminée
                done[key] = True
            else:
                partial[key] += chunk
            progressed = True
        if progressed:
            yield partial['ft'], partial['zs'], done['ft'], done['zs']

    for t in threads.values():
        t.join()

In [ ]:
# Interface Gradio : question en entrée, réponse fine-tunée (+ zero-shot en option) en sortie.
# Pas de sélecteur de langue — le modèle répond dans la langue de la question posée (voir cellule
# précédente) ; tapez simplement votre question dans une des 5 langues du dataset.
#
# Fine-tuné et zero-shot sont désormais générés simultanément sur deux threads (dual_stream,
# cellule précédente), chacun sur son propre modèle en mémoire — chaque panneau se remplit à son
# propre rythme, sans attendre l'autre.
import gradio as gr

PLACEHOLDER = "*(en attente)*"

def ui_fn(question, compare):
    if not question or not question.strip():
        yield "Posez une question ci-dessus, puis cliquez sur Générer.", ""
        return

    prompt = build_prompt(question.strip())
    chat_prompt = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}], tokenize=False, add_generation_prompt=True,
    )

    # Retour immédiat : indispensable, sinon rien ne bouge à l'écran pendant les premières secondes
    # (chargement du prompt, lancement des threads de génération) et le bouton semble ne rien faire.
    yield ("⏳ **Gemma fine-tuné** — génération en cours...",
           "⏳ **Gemma zero-shot (avant fine-tuning)** — génération en cours..." if compare else "")

    t0 = time.time()
    last_ft, last_zs = "", ""
    for partial_ft, partial_zs, done_ft, done_zs in dual_stream(chat_prompt, compare):
        last_ft, last_zs = partial_ft, partial_zs
        elapsed = time.time() - t0

        label_ft = f"**Gemma fine-tuné**" + (f" — généré en {elapsed:.1f} s" if done_ft else "")
        out_ft = f"{label_ft}\n\n{partial_ft}{'' if done_ft else '▌'}"

        if compare:
            label_zs = "**Gemma zero-shot (avant fine-tuning)**" + (f" — généré en {elapsed:.1f} s" if done_zs else "")
            out_zs = f"{label_zs}\n\n{partial_zs}{'' if done_zs else '▌'}"
        else:
            out_zs = ""

        yield out_ft, out_zs


CUSTOM_CSS = """
.gradio-container { max-width: 980px !important; margin: 0 auto; }
#title-md h2 { margin-bottom: 4px; }
#subtitle-md p { color: #6b7280; margin-top: 0; }
.output-panel {
    border: 1px solid #e2ddd0;
    border-radius: 10px;
    padding: 16px 18px;
    background: #fdfcf9;
    min-height: 120px;
}
#panel-finetuned { border-left: 4px solid #1f5e5b; }
#panel-zeroshot  { border-left: 4px solid #b8862e; }
#generate-btn { font-size: 1.05rem; height: 46px; }
"""

with gr.Blocks(title="Démo Gemma — QA santé multilingue", theme=gr.themes.Soft(primary_hue="teal"), css=CUSTOM_CSS) as demo:
    gr.Markdown("## Démo en direct — Gemma E2B + LoRA", elem_id="title-md")
    gr.Markdown(
        "Tapez une question de santé dans n'importe laquelle des 5 langues (anglais, amharique, "
        "luganda, swahili, akan) — le modèle répond dans la même langue, en direct.",
        elem_id="subtitle-md",
    )
    with gr.Group():
        question_box = gr.Textbox(
            label="Question de santé", placeholder="Ex. : Malaria ni nini?", lines=2,
        )
        compare_cb = gr.Checkbox(
            label="Comparer avec la version avant fine-tuning (zero-shot, générée en parallèle)", value=False,
        )
        submit_btn = gr.Button("Générer", variant="primary", elem_id="generate-btn")

    with gr.Row():
        out_ft_md = gr.Markdown(value=PLACEHOLDER, elem_id="panel-finetuned", elem_classes=["output-panel"])
        out_zs_md = gr.Markdown(value="", elem_id="panel-zeroshot", elem_classes=["output-panel"])

    submit_btn.click(ui_fn, inputs=[question_box, compare_cb], outputs=[out_ft_md, out_zs_md])

# share=True : lien public temporaire (~72h), utilisable par le jury depuis son propre appareil pendant
# la soutenance. Lancer cette cellule 10-15 min avant de présenter (temps de chargement du modèle déjà fait
# plus haut ; ce lancement lui-même est quasi instantané).
demo.launch(share=True, debug=False)

---
**Pour la soutenance:** lancer toutes les cellules à l'avance, garder cet onglet Colab ouvert pendant la présentation (le lien `share=True` meurt si la session s'arrête). Prévoir une courte vidéo de secours de la démo qui fonctionne, au cas où le Wi-Fi ou le quota GPU lâche au mauvais moment. Le chargement des deux modèles (fine-tuné + zero-shot) prend plus de mémoire GPU qu'avant — vérifier `L4` ou mieux comme type de GPU Colab, un `T4` (16 Go) est trop juste.

**Amélioration optionnelle plus tard:** une fois `06_merge_adapter.ipynb` terminé, remplacer le chargement du modèle fine-tuné (base 4-bit + `PeftModel` + déballage) par un chargement direct depuis `checkpoints/gemma-4-e2b-merged` — plus rapide, sans dépendance à Hugging Face. Le modèle zero-shot resterait chargé séparément comme aujourd'hui.